In [1]:
# I am upgrading all packages and installing langchain-classic to access legacy chains.
%pip install -U -q langchain langchain-community langchain-core langchain-google-genai python-dotenv pandas langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [4]:
# I am importing the necessary libraries with the correct modern and classic paths.
import os
import sys
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

# I am using the updated core path for the PromptTemplate.
from langchain_core.prompts import PromptTemplate

# I am importing the chains from the new langchain_classic package.
from langchain_classic.chains import LLMChain, SequentialChain

# 1. Setup Environment and API Key
project_root = os.getcwd()
env_path = os.path.join(project_root, '.env')
load_dotenv(dotenv_path=env_path)

if "GOOGLE_API_KEY" not in os.environ:
    print(f"Warning: GOOGLE_API_KEY not found at {env_path}")
else:
    print("API Key loaded successfully.")

# 2) I am initialising the Gemini model using the current active version string.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# 3. Create the Individual Chains
question_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Generate a beginner-level question related to the topic: {topic}"
)
# I am building the first chain to generate the question.
question_chain = LLMChain(llm=llm, prompt=question_prompt, output_key="question")

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="Provide a clear answer with a short explanation for the question: {question}"
)
# I am building the second chain to answer the question.
answer_chain = LLMChain(llm=llm, prompt=answer_prompt, output_key="answer")

# 4. Compose the Sequential Chain
quiz_chain = SequentialChain(
    chains=[question_chain, answer_chain],
    input_variables=["topic"],
    output_variables=["question", "answer"],
    verbose=True
)

# 5. Execute and Display
topic = "Machine Learning Basics"
print(f"\nGenerating quiz for: {topic}...\n")

# I am running the sequential chain.
response = quiz_chain.invoke({"topic": topic})

print("=" * 70)
print(" GENERATED QUIZ")
print("=" * 70)
print("\nQuestion:")
print(response["question"])
print("\nAnswer & Explanation:")
print(response["answer"])
print("\n" + "=" * 70)

API Key loaded successfully.

Generating quiz for: Machine Learning Basics...



> Entering new SequentialChain chain...

> Finished chain.
 GENERATED QUIZ

Question:
Here's a beginner-level question related to Machine Learning Basics:

---

**Question:**

Imagine you want to teach a computer to identify whether a picture contains a 'dog' or a 'cat' using Machine Learning.

1.  What kind of **information (data)** would you primarily need to provide to the machine learning model so it can learn?
2.  In very simple terms, what is the **core idea** of how the model would 'learn' to tell dogs from cats from that information, rather than you explicitly writing down every single rule (like "if it has pointy ears and a long snout, it's a dog")?

---

Answer & Explanation:
Here's a clear answer:

1.  **Information (Data) Needed:** You would primarily need a large collection of **pictures (images)**, with each picture clearly **labeled** as either 'dog' or 'cat'.

2.  **Core Idea of Learning:**

In [11]:
# I am defining the topic for the quiz generator.
topic = "Machine Learning Basics"

# I am executing the sequential chain with the chosen topic.
response = quiz_chain.invoke({"topic": topic})

# I am printing the formatted results.
print("\n" + "="*70)
print(" GENERATED QUIZ")
print("="*70)
print("\nQuestion:")
print(response["question"])
print("\nAnswer & Explanation:")
print(response["answer"])
print("\n" + "="*70)



> Entering new SequentialChain chain...


KeyboardInterrupt: 

In [6]:
# I am importing the modern LangChain Expression Language components.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# I am setting up an output parser to extract clean text from the model's raw output.
parser = StrOutputParser()

# I am defining the individual logic steps using the pipe (|) operator.
generate_question = question_prompt | llm | parser
generate_answer = answer_prompt | llm | parser

# I am building the modern LCEL chain.
# RunnablePassthrough.assign keeps the intermediate 'question' variable in the final dictionary.
lcel_chain = (
    {"topic": RunnablePassthrough()}
    | RunnablePassthrough.assign(question=generate_question)
    | RunnablePassthrough.assign(answer=generate_answer)
)

print("Executing the modern LCEL chain...")
lcel_response = lcel_chain.invoke("Computer Vision Basics")

print("\nQuestion:\n", lcel_response["question"])
print("\nAnswer:\n", lcel_response["answer"])

Executing the modern LCEL chain...

Question:
 Here's a beginner-level question about Computer Vision Basics:

**Question:**

When we look at a digital image (like a photo on a screen), we see colors, shapes, and objects. How does a computer fundamentally 'see' or interpret this image data, given it doesn't have eyes? What is the most basic unit a computer uses to represent the visual information in an image, and what kind of data does this unit typically hold?

Answer:
 A computer fundamentally 'sees' or interprets an image as a **grid of numbers**.

1.  **Most Basic Unit:** The most basic unit a computer uses to represent visual information in an image is a **pixel** (short for "picture element").
2.  **Data Held:** Each pixel typically holds **numerical values representing its color intensity or brightness**.
    *   For a color image (like a JPEG), a pixel usually stores three values: one for **Red**, one for **Green**, and one for **Blue** (known as the RGB model). Each of these v

In [8]:
# I am creating a new prompt template for the hint generation step.
hint_prompt = PromptTemplate(
    input_variables=["question"],
    template="Provide a very brief, subtle hint for this question without giving the actual answer away: {question}"
)

# I am building the hint chain using the classic LLMChain structure.
hint_chain = LLMChain(llm=llm, prompt=hint_prompt, output_key="hint")

# I am composing the new 3-step sequential chain.
# The data will flow: topic -> question -> hint -> answer.
advanced_quiz_chain = SequentialChain(
    chains=[question_chain, hint_chain, answer_chain],
    input_variables=["topic"],
    output_variables=["question", "hint", "answer"],
    verbose=True
)

print("Running the advanced 3-step chain...\n")
advanced_response = advanced_quiz_chain.invoke({"topic": "Data Structures"})

print("\n" + "=" * 70)
print(" ADVANCED QUIZ WITH HINT")
print("=" * 70)
print("\nQuestion:\n", advanced_response["question"])
print("\nHint:\n", advanced_response["hint"])
print("\nAnswer:\n", advanced_response["answer"])
print("\n" + "=" * 70)

Running the advanced 3-step chain...



> Entering new SequentialChain chain...

> Finished chain.

 ADVANCED QUIZ WITH HINT

Question:
 Here's a beginner-level question related to Data Structures:

**Question:**

Imagine you need to store a list of 10 student names in a program, and you know exactly how many names there will be from the start.

a) Which fundamental data structure would be a very common and efficient choice for storing these names?
b) Briefly explain **one reason why** this data structure is a good fit for this scenario.
c) Briefly explain **one potential challenge or limitation** you might face if you later needed to frequently insert or delete names from the *middle* of this list using this same data structure.

Hint:
 Think about how memory is often organized for a fixed, ordered sequence of items.

Answer:
 Here's a clear answer with short explanations:

a) **Array** (or a simple list implementation often backed by an array).

b) **One reason why it's a good fit:**

In [10]:
def interactive_quiz():
    # I am asking for the initial topic.
    topic = input("Enter a topic for your interactive quiz: ")
    print(f"\nGenerating a question about '{topic}'...")

    # I am invoking only the first chain to get the question.
    question_result = question_chain.invoke({"topic": topic})
    question_text = question_result["question"]

    # I am printing the interface directly without clearing the previous output.
    print("\n" + "=" * 70)
    print(" INTERACTIVE QUIZ")
    print("=" * 70)
    print(f"\nQuestion:\n{question_text}\n")

    # I am pausing the script to let the user attempt an answer.
    user_attempt = input("Type your answer here: ")

    print("\nGenerating the comprehensive answer...")

    # I am now running the second chain using the generated question.
    answer_result = answer_chain.invoke({"question": question_text})
    correct_answer = answer_result["answer"]

    # I am printing the final results.
    print("\n" + "=" * 70)
    print(" QUIZ RESULTS")
    print("=" * 70)
    print(f"\nQuestion:\n{question_text}\n")
    print(f"Your Answer:\n{user_attempt}\n")
    print(f"Correct Answer and Explanation:\n{correct_answer}\n")
    print("=" * 70)

# I am executing the function.
interactive_quiz()


Generating a question about 'plutonium'...

 INTERACTIVE QUIZ

Question:
Plutonium is a well-known element, often mentioned in movies and news. What is one major application or use of plutonium that makes it so significant?


Generating the comprehensive answer...

 QUIZ RESULTS

Question:
Plutonium is a well-known element, often mentioned in movies and news. What is one major application or use of plutonium that makes it so significant?

Your Answer:
nuclear technology 


Correct Answer and Explanation:
**Answer:** Nuclear weapons.

**Explanation:** Plutonium is a key fissile material used to create the core of atomic bombs, enabling immense destructive power. This makes it strategically vital and a subject of international concern regarding nuclear proliferation.

